In [1]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import importlib

import os
os.chdir("..")

import models.NN_layers as NN_layers
importlib.reload(NN_layers)
from models.NN_layers import *

import models.Model as Model
importlib.reload(Model)
from models.Model import *

import torch 
import torch.nn as nn
import torch.nn.functional  as F

from torch import Tensor
from torchvision import transforms

import pandas as pd
import PIL.Image as Image

import kagglehub
path = kagglehub.dataset_download("zalando-research/fashionmnist")

c:\Users\katin\OneDrive - Danmarks Tekniske Universitet\6 semester\Fagprojekt\Fagprojekt---Group-Equivariance\fag_projekt\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load billeder

In [48]:
df = pd.read_csv(path + '/fashion-mnist_train.csv') #læs billeder

In [49]:
def format_img(num_images = 1000):
    df_train = df[:num_images]
    img_rows = df_train.iloc[:, 1:].to_numpy() #converts to numpy array (outer dim is pictures)
    img_square = img_rows.reshape(-1, 28,28).astype(np.uint8) # reshape inner dim to be a pictur HxW
    images = torch.tensor(img_square, dtype=torch.complex128).unsqueeze_(1) / 255.0  # make into tensor and scale pixel values to be in range [0,1] instead of [0,255]
    return images

In [50]:
images = format_img(num_images=3)

### Roter billeder til 8 vinkler

In [51]:
def rotate_batch(images, angles_deg):
    images = images.real.to(torch.float64)
    # images: [B, C, H, W]
    B, C, H, W = images.shape
    device = images.device

    out = []

    for angle in angles_deg:
        theta = np.radians(angle)

        # rotation matrix (inverse mapping for grid_sample)
        rot = torch.tensor([
            [np.cos(theta), -np.sin(theta), 0],
            [np.sin(theta),  np.cos(theta), 0]
        ], device=device).unsqueeze(0).repeat(B, 1, 1)

        grid = F.affine_grid(rot, images.size(), align_corners=False)
        rotated = F.grid_sample(images, grid, align_corners=False)
        out.append(rotated)
    return torch.stack(out, dim=1).to(torch.complex128)  # [B, 8, C, H, W]

In [52]:
angles = [i * 45 for i in range(8)]

rotated_images = rotate_batch(images, angles)
print(rotated_images.shape)

torch.Size([3, 8, 1, 28, 28])


### Compute output from model

In [ ]:
model = CNN()

In [62]:
model = NaiveGE_CNN(kernel_size = 5, l = 2, in_features = 1, img_size = 28, n_conv_layers = 2, conv_pr_pool= 1, channels = 8, n_classes= 10, bias= True)

time initiating MLPs: 247.86699795722961
time in creating kernels: 2.744741916656494
time in stacking kernels: 0.031339406967163086
time initiating MLPs: 5.591858148574829
time in creating kernels: 4.378878355026245
time in stacking kernels: 0.030248403549194336


In [60]:
model.model

Sequential(
  (0): LiftingLayer(
    (mlps): ModuleList(
      (0-9): 10 x MLP_Radius(
        (layer): Sequential(
          (0): Linear(in_features=1, out_features=16, bias=True)
          (1): ReLU()
          (2): Linear(in_features=16, out_features=1, bias=True)
        )
      )
    )
  )
  (1): ConvLayer(
    (mlps): ModuleList(
      (0-199): 200 x MLP_Radius(
        (layer): Sequential(
          (0): Linear(in_features=1, out_features=16, bias=True)
          (1): ReLU()
          (2): Linear(in_features=16, out_features=1, bias=True)
        )
      )
    )
  )
  (2): NormNonlinearity()
  (3): AdaptiveAvgPool2d(output_size=(14, 14))
  (4): ConvLayer(
    (mlps): ModuleList(
      (0-799): 800 x MLP_Radius(
        (layer): Sequential(
          (0): Linear(in_features=1, out_features=16, bias=True)
          (1): ReLU()
          (2): Linear(in_features=16, out_features=1, bias=True)
        )
      )
    )
  )
  (5): NormNonlinearity()
  (6): AdaptiveAvgPool2d(output_size=

In [56]:
ls = [16,32,64,128] # layer sizes
kernel_size = 5
l = 2
bias = True
models = [
nn.Sequential(
            LiftingLayer(in_features=1, out_features=ls[0], kernel_size=kernel_size, l=l, bias=bias)
),
nn.Sequential(
            LiftingLayer(in_features=1, out_features=ls[0], kernel_size=kernel_size, l=l, bias=bias)
)
]

In [57]:

#send dem i gennem netværket
#drej dem tilbage og sammenlign.